# **3일차 과제 — 다양한 패턴의 Agent 구현하기**

> **제출자: 조태희** · OZ코딩스쿨 AI 헬스케어 부트캠프 6기 · 자연어 처리(NLP) 3일차 과제 (AI Agent 기초2)

**📌 실행 안내**
- 실행 번호와 출력이 있는 셀은 API 키 없이도 돌아가는 셀(BMI 도구 직접 테스트, 노드 함수 정의, 그래프 조립·구조 확인, 분기 함수 테스트)이라 미리 실행해 둔 결과입니다.
- 코드 첫 줄에 **🔑 [Colab에서 실행]** 표시가 있는 셀은 **OpenAI API 키와 Tavily API 키**가 필요해서 아직 실행하지 않았습니다. Colab에서 `api_key.txt`(두 키가 한 줄씩)를 업로드하거나 왼쪽 🔑 보안 비밀에 `OPENAI_API_KEY`, `TAVILY_API_KEY`를 등록한 뒤 **런타임 → 모두 실행**으로 위에서부터 차례대로 실행해 주세요.
- GPT 답변과 검색 결과는 실행할 때마다 조금씩 달라질 수 있습니다.

# **1. 환경준비**

## **(1) 라이브러리**

In [ ]:
# 🔑 [Colab에서 실행] 필요한 라이브러리 설치 (-q: 설치 로그를 짧게 표시, langgraph는 의존성으로 함께 설치됨)
!pip install langchain-openai langchain-tavily -q

In [1]:
import os         # 환경변수(API 키) 관리
import operator   # operator.add → 리스트 '쌓기' 규칙(리듀서)으로 사용

from langchain_openai import ChatOpenAI                          # OpenAI GPT 모델
from langchain_core.messages import HumanMessage, SystemMessage  # 메시지 종류 (사용자 요청 / 역할·규칙 지침)
from langchain_core.tools import tool                            # 파이썬 함수를 LLM이 쓸 수 있는 '도구'로 만드는 데코레이터
from langchain_tavily import TavilySearch                        # Tavily 웹 검색 도구
from langgraph.graph import StateGraph, START, END               # 그래프 설계도, 시작점, 끝점
from langgraph.graph.message import MessagesState                # messages 칸(쌓기)이 미리 들어 있는 기본 State
from langgraph.prebuilt import ToolNode, tools_condition         # 도구 실행 노드, 도구 호출 여부로 분기하는 함수
from langgraph.checkpoint.memory import MemorySaver              # 대화 State를 저장해 두는 메모리(체크포인터)

## **(2) API Key 확인**

In [ ]:
# 🔑 [Colab에서 실행] api_key.txt 파일 업로드
#   - api_key.txt 안에는 아래처럼 한 줄에 하나씩 적혀 있어야 합니다.
#       OPENAI_API_KEY=sk-...
#       TAVILY_API_KEY=tvly-...
#   - Colab 왼쪽 🔑(보안 비밀)에 두 키를 등록해 두었다면 이 셀은 건너뛰어도 됩니다.
from google.colab import files   # Colab에서 내 컴퓨터 파일을 올리는 기능
uploaded = files.upload()        # 업로드 창이 뜨면 api_key.txt를 선택합니다

In [ ]:
# 🔑 [Colab에서 실행] API 키를 환경변수로 등록합니다 (코드에 키를 직접 적지 않기 위해서예요)
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:                      # 키 파일을 읽기 모드로 엽니다
        for line in f:                                  # 한 줄씩 읽으면서
            line = line.strip()                         # 앞뒤 공백·줄바꿈을 지우고
            if line and "=" in line:                    # 빈 줄이 아니고 "이름=값" 형태라면
                key, value = line.split("=", 1)         # 첫 번째 '='를 기준으로 이름과 값을 나눠서
                os.environ[key.strip()] = value.strip() # 환경변수로 등록합니다

def ensure_api_key(name):
    """환경변수에 키가 없으면 Colab 보안 비밀 → 직접 입력 순서로 채웁니다."""
    if os.environ.get(name):                            # 이미 등록돼 있으면 그대로 사용
        return                                          # 더 할 일이 없으니 함수 종료
    try:
        from google.colab import userdata               # Colab 🔑 보안 비밀 저장소
        os.environ[name] = userdata.get(name)           # 보안 비밀에 등록된 키를 꺼내 등록
    except Exception:                                   # 보안 비밀에 없거나 Colab이 아니면
        from getpass import getpass                     # 입력한 글자가 화면에 안 보이는 입력창
        os.environ[name] = getpass(f"{name}를 입력하세요: ")   # 직접 입력받아 등록


if os.path.exists("api_key.txt"):                       # 업로드한 키 파일이 있으면
    load_api_keys("api_key.txt")                        # 파일에서 읽어서 등록
ensure_api_key("OPENAI_API_KEY")                        # OpenAI 키 확인 (없으면 보안 비밀/직접 입력)
ensure_api_key("TAVILY_API_KEY")                        # Tavily 검색 키 확인 (문제 2에서 사용)

* 아래 코드셀은, 실행해서 key가 제대로 보이는지 확인하고 삭제하세요.

In [ ]:
# 🔑 [Colab에서 실행] 키가 잘 들어왔는지 앞부분만 확인합니다
#   (전체 키가 노트북에 남으면 유출될 수 있어서 앞 7글자만 보이게 했습니다. 확인 후 이 셀은 삭제하세요.)
print(os.environ['OPENAI_API_KEY'][:7] + "...")
print(os.environ['TAVILY_API_KEY'][:7] + "...")

In [ ]:
# 🔑 [Colab에서 실행] GPT 모델 준비 (API 키가 있어야 객체를 만들 수 있습니다)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)   # temperature=0 → 답이 흔들리지 않게 무작위성 최소화

# **2. 과제**

## 문제 1. BMI 계산 도구를 활용하는 건강관리 Agent (커스텀 도구)

* 키(cm)와 몸무게(kg)를 입력하면 BMI를 계산해주는 커스텀 도구를 만들고, 이를 활용하는 Agent를 구성하세요.
* 입력 예시
    * `"키 170cm, 몸무게 68kg인데 체질량지수 좀 계산해줘"`
* 요구사항
    * `@tool` 데코레이터와 docstring을 갖춘 `bmi_calculator` 함수 작성 (BMI = 몸무게(kg) / 키(m)^2)
    * `llm.bind_tools([bmi_calculator])`로 도구 연결
    * `ToolNode`, `tools_condition`으로 그래프 구성 후 실행

**💡 풀이 포인트** — **도구**(tool)는 LLM이 스스로 잘 못 하는 일(정확한 계산, 검색 등)을 대신 해주는 파이썬 함수입니다. `@tool`을 붙이면 함수 이름·docstring(함수 설명문)·인자 타입이 LLM에게 **도구 설명서**로 전달되기 때문에, docstring을 구체적으로 써야 LLM이 "언제 이 도구를 쓸지" 판단할 수 있어요. `bind_tools`는 LLM에게 이 설명서를 쥐여 주는 단계이고, LLM이 도구를 쓰겠다고 답하면(`tool_calls`) `tools_condition`이 그래프를 `ToolNode`(도구를 실제로 실행하는 노드)로 보내고, 계산 결과를 받은 LLM이 최종 답변을 만듭니다. BMI 판정 구간은 대한비만학회 기준(정상 18.5~22.9, 비만 전단계 23~24.9, 1단계 비만 25~29.9, 2단계 비만 30~34.9, 3단계 비만 35 이상)을 사용했습니다.

In [2]:
# [문제 1] 커스텀 도구 정의 — docstring이 LLM에게 전달되는 '도구 설명서'가 됩니다
@tool
def bmi_calculator(height_cm: float, weight_kg: float) -> str:
    """키(cm)와 몸무게(kg)를 받아 체질량지수(BMI)를 계산하고 비만도 판정을 알려줍니다.
    사용자가 BMI나 체질량지수를 물어보면 이 도구를 사용하세요.

    Args:
        height_cm: 키 (센티미터 단위, 예: 170)
        weight_kg: 몸무게 (킬로그램 단위, 예: 68)
    """
    height_m = height_cm / 100            # cm → m 로 단위 변환 (170cm → 1.7m)
    bmi = weight_kg / (height_m ** 2)     # BMI 공식: 몸무게(kg) ÷ 키(m)의 제곱

    # 대한비만학회 기준으로 판정
    if bmi < 18.5:
        category = "저체중"   # 18.5 미만
    elif bmi < 23:
        category = "정상"   # 18.5 이상 23 미만
    elif bmi < 25:
        category = "비만 전단계(과체중)"   # 23 이상 25 미만
    elif bmi < 30:
        category = "1단계 비만"   # 25 이상 30 미만
    elif bmi < 35:
        category = "2단계 비만"   # 30 이상 35 미만
    else:
        category = "3단계 비만"   # 35 이상
    return f"BMI는 {bmi:.1f}이며, 판정은 '{category}'입니다."   # 소수점 첫째 자리까지 표시


# 도구가 LLM에게 어떻게 보이는지 확인 (이름 / 인자 정보)
print("도구 이름:", bmi_calculator.name)
print("인자 정보:", bmi_calculator.args)

# LLM 없이 도구만 직접 실행해서 계산이 맞는지 확인 (68 ÷ 1.7² = 23.53...)
print(bmi_calculator.invoke({"height_cm": 170, "weight_kg": 68}))

도구 이름: bmi_calculator
인자 정보: {'height_cm': {'title': 'Height Cm', 'type': 'number'}, 'weight_kg': {'title': 'Weight Kg', 'type': 'number'}}
BMI는 23.5이며, 판정은 '비만 전단계(과체중)'입니다.


In [ ]:
# 🔑 [Colab에서 실행] LLM에 도구를 연결합니다 (API 키 필요)
llm_with_bmi = llm.bind_tools([bmi_calculator])   # LLM에게 'bmi_calculator 도구 설명서'를 쥐여 줌

# (확인) 도구를 연결한 LLM이 이 질문에 '도구를 쓰겠다'고 하는지 살펴보기
test_msg = llm_with_bmi.invoke("키 170cm, 몸무게 68kg인데 체질량지수 좀 계산해줘")   # 질문 하나를 바로 보내 보기
print(test_msg.tool_calls)   # [{'name': 'bmi_calculator', 'args': {...}, ...}] 형태면 도구 호출을 요청한 것

In [3]:
# Agent 노드: 대화를 보고 도구를 쓸지, 바로 답할지 LLM이 결정합니다
def bmi_agent(state: MessagesState):
    s_msg = SystemMessage(content="당신은 친절한 건강관리 도우미입니다. BMI(체질량지수) 계산이 필요하면 반드시 "
                                  "bmi_calculator 도구를 사용하고, 도구 결과를 바탕으로 쉽게 설명해 주세요.")
    response = llm_with_bmi.invoke([s_msg] + state["messages"])   # 지침 + 지금까지의 대화(도구 결과 포함)
    return {"messages": [response]}                                # LLM 응답(도구 호출 요청 또는 최종 답변)을 쌓기

In [4]:
builder = StateGraph(MessagesState)                     # messages 칸이 기본으로 있는 State 사용
builder.add_node("agent", bmi_agent)                    # LLM이 판단하는 노드
builder.add_node("tools", ToolNode([bmi_calculator]))   # 도구를 실제로 실행하는 노드 (이름을 "tools"로 해야 tools_condition과 맞음)
builder.add_edge(START, "agent")                        # 시작 → agent
builder.add_conditional_edges("agent", tools_condition) # tool_calls가 있으면 "tools"로, 없으면 END로
builder.add_edge("tools", "agent")                      # 도구 결과를 들고 다시 agent로 → 최종 답변 작성
bmi_graph = builder.compile()                           # 완성

# 그래프 연결 구조 확인 (__start__ = START, __end__ = END)
drawn = bmi_graph.get_graph()                                              # 그래프 구조(노드·엣지) 정보 꺼내기
node_order = list(drawn.nodes)                                       # 노드를 등록한 순서
for edge in sorted(drawn.edges, key=lambda e: node_order.index(e.source)):   # 출발 노드 순서대로 정렬해서 하나씩
    label = ""                                                       # 기본값: 조건 표시 없음
    if edge.conditional:                                             # 조건부 연결이면 어떤 조건인지 표시
        label = f"   (조건부: {edge.data})" if edge.data else "   (조건부)"
    print(edge.source, "→", edge.target, label)

__start__ → agent 
agent → __end__    (조건부)
agent → tools    (조건부)
tools → agent 


In [ ]:
# 🔑 [Colab에서 실행] Agent 실행 (API 키 필요)
result1 = bmi_graph.invoke({"messages": [HumanMessage(content="키 170cm, 몸무게 68kg인데 체질량지수 좀 계산해줘")]})   # 질문을 넣어 Agent 실행

# 메시지 흐름 전체 출력: Human → AI(도구 호출 요청) → Tool(계산 결과) → AI(최종 답변)
for m in result1["messages"]:
    m.pretty_print()   # 메시지 종류별로 보기 좋게 출력

## 문제 2. 최신 건강정보 검색 Agent (Tavily 도구)

* 관심 있는 질환이나 건강 키워드를 입력하면, Tavily로 최신 관리법/생활습관 정보를 검색해 답변하는 Agent를 만드세요.
* 입력 예시
    * `"고혈압에 좋은 생활습관 알려줘"`
* 요구사항
    * `TavilySearch`를 `.bind_tools()`로 llm에 연결
    * `ToolNode`, `tools_condition`으로 도구 호출 여부를 그래프에서 분기
    * SystemMessage에 "최신 정보가 필요하면 검색 도구를 활용하라"는 지침 명시

**💡 풀이 포인트** — Tavily는 LLM에서 쓰기 좋게 만든 웹 검색 API입니다. GPT는 학습한 시점 이후의 정보를 모르기 때문에, 시스템 메시지에 "최신 정보가 필요하면 검색 도구를 활용하라"고 적어 두면 필요할 때 검색을 요청(`tool_calls`)합니다. 그래프 구조는 문제 1과 똑같고 **도구만 바뀌었는데**, 이렇게 도구만 갈아 끼우면 다른 기능의 Agent가 된다는 게 Agent 패턴의 장점이에요. Tavily 객체는 만들 때 키를 확인하므로 아래 첫 셀부터 키가 필요합니다.

In [ ]:
# 🔑 [Colab에서 실행] Tavily 검색 도구 준비 + LLM에 연결 (Tavily·OpenAI 키 필요)
tavily_tool = TavilySearch(max_results=3)      # 검색 결과를 최대 3개까지 가져오는 검색 도구
search_tools = [tavily_tool]                   # 도구 목록 (나중에 도구를 더 넣을 수도 있음)
llm_with_search = llm.bind_tools(search_tools) # LLM에게 검색 도구 설명서를 쥐여 줌

In [5]:
# 검색 Agent 노드: 시스템 메시지에 '최신 정보가 필요하면 검색 도구 활용' 지침을 명시
def search_agent(state: MessagesState):
    s_msg = SystemMessage(content="당신은 건강 정보를 알려주는 상담 도우미입니다. "
                                  "최신 정보가 필요하면 검색 도구를 활용하라. "
                                  "검색 결과를 바탕으로 핵심 관리법을 쉽게 정리하고, 의학적 진단을 대신하지 않는다는 점을 함께 안내하세요.")
    response = llm_with_search.invoke([s_msg] + state["messages"])   # 지침 + 대화(검색 결과 포함)
    return {"messages": [response]}                                   # 검색 요청 또는 최종 답변을 쌓기

In [ ]:
# ▶ [Colab에서 실행] 바로 위의 Tavily 도구 셀(search_tools)을 먼저 실행해야 그래프를 만들 수 있습니다
builder = StateGraph(MessagesState)                       # messages 기본 State
builder.add_node("agent", search_agent)                   # 검색 여부를 판단하는 LLM 노드
builder.add_node("tools", ToolNode(search_tools))         # 검색을 실제로 실행하는 노드
builder.add_edge(START, "agent")                          # 시작 → agent
builder.add_conditional_edges("agent", tools_condition)   # 검색 요청이 있으면 tools, 없으면 END
builder.add_edge("tools", "agent")                        # 검색 결과를 들고 다시 agent로
search_graph = builder.compile()                          # 완성

# 그래프 연결 구조 확인 (__start__ = START, __end__ = END)
drawn = search_graph.get_graph()                                              # 그래프 구조(노드·엣지) 정보 꺼내기
node_order = list(drawn.nodes)                                       # 노드를 등록한 순서
for edge in sorted(drawn.edges, key=lambda e: node_order.index(e.source)):   # 출발 노드 순서대로 정렬해서 하나씩
    label = ""                                                       # 기본값: 조건 표시 없음
    if edge.conditional:                                             # 조건부 연결이면 어떤 조건인지 표시
        label = f"   (조건부: {edge.data})" if edge.data else "   (조건부)"
    print(edge.source, "→", edge.target, label)

In [ ]:
# 🔑 [Colab에서 실행] 검색 Agent 실행 (Tavily·OpenAI 키 필요)
result2 = search_graph.invoke({"messages": [HumanMessage(content="고혈압에 좋은 생활습관 알려줘")]})   # 질문을 넣어 검색 Agent 실행

# LLM이 어떤 검색어로 검색했는지 확인
for m in result2["messages"]:
    for call in getattr(m, "tool_calls", None) or []:      # 도구 호출 요청이 있는 메시지만
        print("🔎 검색어:", call["args"].get("query"))

print("\n[최종 답변]")
print(result2["messages"][-1].content)                     # 마지막 메시지 = 검색 결과를 반영한 최종 답변

## 문제 3. 증상 요약 Agent + 검토 Agent 반복 루프 (Loop)

* 장황한 환자 증상 설명을 요약하고, 요약 품질을 검토해서 부적합하면 다시 요약하는 Agent를 만드세요.
* 입력 예시
    * `"한 3일 전부터 그런 것 같은데, 처음엔 그냥 가볍게 콜록거리는 정도였거든요. 근데 어제 저녁부터 갑자기 열이 좀 나기 시작하더니 오늘 아침엔 목이 붓고 침 삼킬 때마다 아프고, 가래도 누런색으로 나오고 있어요. 몸살 기운도 좀 있는 것 같고 으슬으슬 춥기도 해요."`
* 처리 흐름
    * `summarize_agent` : 증상을 3문장 이내로 요약 → `summary`
    * `review_agent` : 요약문이 3문장 이내이고 핵심 증상(기침/발열/목 통증 등)을 빠짐없이 담았는지 검토 → `satisfied`("적합"/"부적합")
    * 부적합이면 `summarize_agent`로 돌아가 재생성 (최대 2회)
* 요구사항
    * State: `messages`(쌓기), `symptom_text`(덮어쓰기), `summary`(덮어쓰기), `satisfied`(덮어쓰기), `loop_count`(덮어쓰기)
    * `review_agent`에서 `loop_count + 1`을 State에 반영
    * 조건 분기 함수에서 `satisfied == "적합"`이거나 `loop_count`가 2 이상이면 종료(`done`), 아니면 재시도(`retry`)

**💡 풀이 포인트** — **루프**(loop)는 "검토 → 불합격이면 다시"를 반복하는 구조로, 조건부 엣지가 앞쪽 노드를 다시 가리키게 만들면 됩니다. 대신 무한 반복을 막는 **탈출 조건**이 꼭 필요해서, `review_agent`가 검토할 때마다 `loop_count`를 1씩 올리고 분기 함수가 "적합이거나 loop_count ≥ 2면 done"으로 끝냅니다. 이 규칙이면 요약은 최대 2번(처음 1번 + 재생성 1번) 만들어져요. 재요약할 때는 이전 요약을 함께 보여줘서 "무엇을 고쳐야 하는지" 알게 했고, 파이썬에서 `"적합" in "부적합"`이 True가 되는 함정이 있어서 **'부적합'을 먼저 검사**했습니다.

In [6]:
from typing import TypedDict, Annotated   # State 정의 도구 (이 노트북 맨 위 import에는 없어서 여기서 불러옴)

# [문제 3] 요약-검토 루프용 State
class SummaryState(TypedDict):
    messages: Annotated[list, operator.add]   # 메시지 기록 → 쌓기
    symptom_text: str                          # 원래 증상 설명 → 덮어쓰기
    summary: str                               # 요약문 → 덮어쓰기 (재생성하면 새 요약으로 교체)
    satisfied: str                             # 검토 결과 "적합"/"부적합" → 덮어쓰기
    loop_count: int                            # 검토 횟수 → 덮어쓰기 (탈출 조건에 사용)

In [7]:
def summarize_agent(state: SummaryState):
    s_msg = SystemMessage(content="""당신은 환자의 장황한 증상 설명을 의료진이 보기 좋게 정리하는 간호사입니다.
- 반드시 3문장 이내로 요약하세요.
- 기침, 발열, 목 통증, 가래, 몸살·오한 등 원문에 나온 핵심 증상과 시작 시점을 빠짐없이 담으세요.
- 원문에 없는 내용은 추가하지 마세요.""")
    request = f"[증상 설명]\n{state['symptom_text']}"                   # 기본 요청: 원문 요약
    if state.get("satisfied") == "부적합":                               # 이전 요약이 검토에서 탈락했다면
        request += (f"\n\n[이전 요약 — 검토 결과 부적합]\n{state['summary']}\n"
                    "위 요약에서 빠진 증상이나 문장 수 문제를 보완해서 다시 요약하세요.")   # 고칠 점을 알려줌
    response = llm.invoke([s_msg, HumanMessage(content=request)])         # GPT 호출 (요약 요청)
    return {"messages": [response], "summary": response.content}          # 새 요약으로 summary 덮어쓰기


def review_agent(state: SummaryState):
    s_msg = SystemMessage(content="""당신은 의료 기록 품질 검토자입니다. [원문]과 [요약문]을 비교해서 아래 두 기준을 검토하세요.
1) 요약문이 3문장 이내인가?
2) 원문의 핵심 증상(기침, 발열, 목 통증, 가래 등)을 빠짐없이 담았는가?
두 기준을 모두 만족하면 "적합", 하나라도 만족하지 못하면 "부적합"이라고만 답하세요. 다른 말은 절대 붙이지 마세요.""")
    h_msg = HumanMessage(content=f"[원문]\n{state['symptom_text']}\n\n[요약문]\n{state['summary']}")   # 원문과 요약문을 함께 보여 줌
    response = llm.invoke([s_msg, h_msg])                                 # GPT에게 검토 요청
    # ⚠️ '부적합'이라는 글자 안에 '적합'이 들어 있으므로 반드시 '부적합'을 먼저 확인!
    satisfied = "부적합" if "부적합" in response.content else "적합"
    return {"messages": [response],
            "satisfied": satisfied,                                       # 검토 결과 저장
            "loop_count": state["loop_count"] + 1}                        # 검토 횟수 +1 (요구사항)

In [8]:
def check_review(state: SummaryState) -> str:
    # 적합 판정이거나, 이미 2번 검토했으면 종료 / 아니면 다시 요약
    if state["satisfied"] == "적합" or state["loop_count"] >= 2:
        return "done"
    return "retry"


builder = StateGraph(SummaryState)                       # 새 그래프 설계도
builder.add_node("summarize_agent", summarize_agent)     # 요약 노드
builder.add_node("review_agent", review_agent)           # 검토 노드
builder.add_edge(START, "summarize_agent")               # 시작 → 요약
builder.add_edge("summarize_agent", "review_agent")      # 요약 → 검토
builder.add_conditional_edges(
    "review_agent",                                      # 검토가 끝나면
    check_review,                                        # 이 함수로 판단해서
    {"retry": "summarize_agent",                         # retry → 다시 요약 노드로 (루프!)
     "done": END},                                       # done  → 종료
)
loop_graph = builder.compile()                           # 완성

# 그래프 연결 구조 확인 (__start__ = START, __end__ = END)
drawn = loop_graph.get_graph()                                              # 그래프 구조(노드·엣지) 정보 꺼내기
node_order = list(drawn.nodes)                                       # 노드를 등록한 순서
for edge in sorted(drawn.edges, key=lambda e: node_order.index(e.source)):   # 출발 노드 순서대로 정렬해서 하나씩
    label = ""                                                       # 기본값: 조건 표시 없음
    if edge.conditional:                                             # 조건부 연결이면 어떤 조건인지 표시
        label = f"   (조건부: {edge.data})" if edge.data else "   (조건부)"
    print(edge.source, "→", edge.target, label)

# 분기 함수만 따로 테스트 (LLM 없이, 직접 만든 State로 확인)
print("\n적합 & 1회차   →", check_review({"satisfied": "적합", "loop_count": 1}))     # done (통과)
print("부적합 & 1회차 →", check_review({"satisfied": "부적합", "loop_count": 1}))   # retry (다시 요약)
print("부적합 & 2회차 →", check_review({"satisfied": "부적합", "loop_count": 2}))   # done (최대 횟수 → 종료)

__start__ → summarize_agent 
summarize_agent → review_agent 
review_agent → __end__    (조건부: done)
review_agent → summarize_agent    (조건부: retry)

적합 & 1회차   → done
부적합 & 1회차 → retry
부적합 & 2회차 → done


In [ ]:
# 🔑 [Colab에서 실행] 루프 그래프 실행 — 어떤 노드가 어떤 순서로 실행됐는지도 출력합니다 (API 키 필요)
symptom = ("한 3일 전부터 그런 것 같은데, 처음엔 그냥 가볍게 콜록거리는 정도였거든요. "
           "근데 어제 저녁부터 갑자기 열이 좀 나기 시작하더니 오늘 아침엔 목이 붓고 침 삼킬 때마다 아프고, "
           "가래도 누런색으로 나오고 있어요. 몸살 기운도 좀 있는 것 같고 으슬으슬 춥기도 해요.")

inputs = {"symptom_text": symptom, "loop_count": 0, "messages": []}   # 처음 State (검토 횟수는 0부터)
final_state = None                                                     # 최종 State를 담을 변수 (아직 없음)
# stream: 노드가 하나 끝날 때마다 결과를 받아볼 수 있는 실행 방법
for mode, chunk in loop_graph.stream(inputs, stream_mode=["updates", "values"]):
    if mode == "updates":                                    # updates = 방금 실행된 노드가 바꾼 칸
        for node_name, update in chunk.items():
            info = {k: update[k] for k in ("satisfied", "loop_count") if k in update}   # 검토 결과만 골라 표시
            print(f"▶ {node_name} 실행 {info if info else ''}")
    else:                                                    # values = 그 시점의 전체 State
        final_state = chunk                                  # 마지막으로 받은 값이 최종 State

print("\n[최종 요약]\n" + final_state["summary"])
print("\n검토 결과:", final_state["satisfied"], "| 검토 횟수(loop_count):", final_state["loop_count"])

## 문제 4. 개인 맞춤형 상담 챗봇 (메모리 추가)

* 사용자의 이름과 선호(식습관 등)를 대화 중 기억했다가, 이후 대화에서 활용해 답변하는 Agent를 만드세요.
* 입력 예시
    * 1턴: `"내 이름은 지영이고, 매운 음식을 잘 못 먹어"`
    * 2턴: `"오늘 저녁 메뉴 추천해줘"`
* 요구사항
    * `MemorySaver`를 이용해 `checkpointer`와 함께 그래프 컴파일
    * 동일한 `thread_id`로 두 메시지를 순서대로 실행
    * 두 번째 답변에서 첫 번째 메시지의 정보가 반영되는지 확인

**💡 풀이 포인트** — `MemorySaver`는 그래프가 한 단계씩 실행될 때마다 그 시점의 State를 저장해 두는 **체크포인터**(checkpointer, 저장 장치)입니다. 같은 `thread_id`(대화방 번호)로 다시 실행하면 저장된 대화 뒤에 새 메시지가 이어 붙기 때문에, 2일차 문제 2처럼 이전 messages를 직접 넘겨줄 필요가 없어요. 마지막에 **다른 thread_id**로도 물어봐서, 대화방이 다르면 기억이 공유되지 않는다는 것까지 확인합니다.

In [9]:
# 맞춤형 상담 챗봇 노드: 대화 중 알게 된 이름·선호를 기억해서 반영하도록 지시
def memory_chatbot(state: MessagesState):
    s_msg = SystemMessage(content="당신은 사용자의 건강과 식습관을 고려해 조언하는 친절한 개인 맞춤형 상담 챗봇입니다. "
                                  "대화 중 사용자가 알려준 이름과 선호(식습관, 못 먹는 음식 등)를 기억했다가 "
                                  "이후 답변에서 이름을 불러 주고 선호를 반드시 반영하세요.")
    response = llm.invoke([s_msg] + state["messages"])   # 지침 + (메모리에서 불러온) 이전 대화 + 새 메시지
    return {"messages": [response]}                       # 답변을 쌓기 → 체크포인터가 저장

In [10]:
memory = MemorySaver()                                   # 대화 State를 저장할 메모리(체크포인터)

builder = StateGraph(MessagesState)                      # messages 기본 State
builder.add_node("chatbot", memory_chatbot)              # 챗봇 노드
builder.add_edge(START, "chatbot")                       # 시작 → 챗봇
builder.add_edge("chatbot", END)                         # 챗봇 → 끝
memory_graph = builder.compile(checkpointer=memory)      # ⭐ checkpointer를 넣어서 컴파일 → 실행 단계마다 State가 저장됨

config = {"configurable": {"thread_id": "jiyoung_1"}}    # 대화방 번호 (같은 번호 = 같은 대화로 이어짐)

# 그래프 연결 구조 확인 (__start__ = START, __end__ = END)
drawn = memory_graph.get_graph()                                              # 그래프 구조(노드·엣지) 정보 꺼내기
node_order = list(drawn.nodes)                                       # 노드를 등록한 순서
for edge in sorted(drawn.edges, key=lambda e: node_order.index(e.source)):   # 출발 노드 순서대로 정렬해서 하나씩
    label = ""                                                       # 기본값: 조건 표시 없음
    if edge.conditional:                                             # 조건부 연결이면 어떤 조건인지 표시
        label = f"   (조건부: {edge.data})" if edge.data else "   (조건부)"
    print(edge.source, "→", edge.target, label)

__start__ → chatbot 
chatbot → __end__ 


In [ ]:
# 🔑 [Colab에서 실행] 1턴 — 이름과 식습관 알려주기 (API 키 필요)
result_t1 = memory_graph.invoke({"messages": [HumanMessage(content="내 이름은 지영이고, 매운 음식을 잘 못 먹어")]},
                                config=config)   # 같은 config(thread_id)로 실행해야 기억이 이어짐
print(result_t1["messages"][-1].content)         # 1턴 답변

In [ ]:
# 🔑 [Colab에서 실행] 2턴 — 같은 thread_id로 실행하면 1턴 대화를 기억합니다 (API 키 필요)
result_t2 = memory_graph.invoke({"messages": [HumanMessage(content="오늘 저녁 메뉴 추천해줘")]},
                                config=config)   # 새 메시지만 넣어도 저장된 1턴 대화가 앞에 자동으로 붙음
answer_t2 = result_t2["messages"][-1].content   # 마지막 메시지 = 2턴 답변
print(answer_t2)

# 1턴 정보가 반영됐는지 간단히 확인
print("\n[확인] 이름(지영) 언급:", "지영" in answer_t2)
print("[확인] 매운 음식 관련 언급:", any(w in answer_t2 for w in ["매운", "맵", "자극"]))
print("[확인] 저장된 메시지 수:", len(memory_graph.get_state(config).values["messages"]))   # 1턴 2개 + 2턴 2개 = 4개

# (참고) 다른 thread_id = 다른 대화방 → 1턴 내용을 모릅니다
other = memory_graph.invoke({"messages": [HumanMessage(content="내 이름이 뭐였지?")]},
                            config={"configurable": {"thread_id": "other_user"}})
print("\n[다른 대화방에서 '내 이름이 뭐였지?'] →", other["messages"][-1].content)